# Week 12 과제 · 탐색적 데이터 분석과 프로젝트 기획

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week12/assignment.ipynb)

## 제출 방법

1. **파일 → 드라이브에 사본 저장** 후 파일 이름을 `week12_학번_이름.ipynb`로 바꿉니다.
2. `# TODO` 부분을 채우고 각 문제 아래 **자동 확인 셀**이 ✅인지 확인합니다.
3. 문제별 **AI 활용 기록**, **최종 프로젝트 제안서**, 마지막 **성찰**을 작성합니다.
4. **런타임 → 다시 시작 및 모두 실행** 후 `.ipynb` 파일을 제출합니다.

| 문제 | 내용 | 배점 |
|---|---|---:|
| 1 | 시간 파생 변수 만들기 | 2 |
| 2 | 요일 × 시간대 일평균 매출표 | 2 |
| 3 | 메뉴 매출 비중과 상위 80% 메뉴 | 3 |
| 4 | 기온 구간 분석 (AI 협업) + 프로젝트 제안서 | 3 |
| | **합계** | **10** |

## 준비: 데이터와 자동 확인 도우미

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

CAFE = pd.read_csv(get_data("cafe_sales.csv"))           # date는 아직 문자열
WEATHER = pd.read_csv(get_data("weather_daily.csv"))
CAFE_BACKUP = CAFE.copy()
WEEKDAYS = ["월", "화", "수", "목", "금", "토", "일"]
DAYPARTS = ["아침", "점심", "오후", "저녁"]


def check(desc, test):
    """자동 확인 도우미: test는 True/False를 돌려주는 함수(lambda)입니다."""
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ 미작성 - {desc}")
    except Exception as e:
        print(f"❌ 오류 - {desc} ({type(e).__name__}: {e})")
    else:
        print(f"{'✅ 통과' if ok else '❌ 실패'} - {desc}")

---
## 문제 1. 시간 파생 변수 만들기 (2점)

`add_features(cafe)`는 원본을 바꾸지 않고 다음 열을 추가(또는 변환)한 새 DataFrame을 돌려줍니다.

| 열 | 내용 |
|---|---|
| `date` | 문자열 → `datetime` |
| `revenue` | `unit_price × qty` |
| `month` | 월 (정수) |
| `weekday` | `"월"`~`"일"` 한 글자 |
| `hour` | `time`의 시 (정수) |
| `daypart` | 8~10시 `"아침"`, 11~13시 `"점심"`, 14~16시 `"오후"`, 17~19시 `"저녁"` (문자열) |

In [ ]:
def add_features(cafe):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
check("새 열", lambda: {"revenue", "month", "weekday", "hour", "daypart"} <= set(add_features(CAFE).columns))
check("date는 datetime", lambda: add_features(CAFE)["date"].dtype.kind == "M")
check("매출 합계", lambda: add_features(CAFE)["revenue"].sum() == 14_198_000)
check("요일 값", lambda: set(add_features(CAFE)["weekday"]) == set("월화수목금토"))
check("시간대 경계 (10시=아침, 11시=점심, 19시=저녁)", lambda: add_features(CAFE).groupby("hour")["daypart"].first()[[10, 11, 16, 17, 19]].tolist()
      == ["아침", "점심", "오후", "저녁", "저녁"])
check("원본 불변", lambda: (add_features(CAFE), CAFE.equals(CAFE_BACKUP))[1])

---
## 문제 2. 요일 × 시간대 일평균 매출표 (2점)

`weekday_daypart(cafe)`는 **요일(행, 월~토 순서) × 시간대(열, 아침·점심·오후·저녁 순서)** 의 **일평균 매출(원, 정수 반올림)** 표를 돌려줍니다.

- 일평균 = 그 칸의 매출 합계 ÷ **그 요일의 영업일 수** (데이터에 그 요일이 나온 날짜 수)
- 문제 1의 `add_features`를 재사용하세요.

In [ ]:
def weekday_daypart(cafe):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
check("모양과 순서", lambda: list(weekday_daypart(CAFE).index) == list("월화수목금토")
      and list(weekday_daypart(CAFE).columns) == DAYPARTS)
check("월요일 점심 일평균", lambda: weekday_daypart(CAFE).loc["월", "점심"] == 65_888)
check("토요일 저녁 일평균", lambda: weekday_daypart(CAFE).loc["토", "저녁"] == 8_473)
check("가장 바쁜 칸은 화요일 점심", lambda: weekday_daypart(CAFE).stack().idxmax() == ("화", "점심"))

---
## 문제 3. 메뉴 매출 비중과 상위 80% 메뉴 (3점)

1. `menu_share(cafe)`: 메뉴를 인덱스로, **매출 내림차순** 정렬. 열은
   - `qty`(판매 수량), `revenue`(매출), `share`(매출 비중 %, 소수 첫째 자리), `cum_share`(누적 비중 %, 소수 첫째 자리)
   - `cum_share`는 반올림 **전** 비중을 누적한 뒤 반올림합니다.
2. `top80(cafe)`: 누적 비중이 **처음으로 80% 이상이 되는 메뉴까지**의 메뉴 이름 리스트 (매출 순)

In [ ]:
def menu_share(cafe):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")


def top80(cafe):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
check("열 구성과 정렬", lambda: list(menu_share(CAFE).columns) == ["qty", "revenue", "share", "cum_share"]
      and menu_share(CAFE)["revenue"].is_monotonic_decreasing)
check("1위 메뉴와 비중", lambda: (menu_share(CAFE).index[0], menu_share(CAFE)["share"].iloc[0]) == ("아메리카노", 23.6))
check("누적 비중 마지막 100", lambda: menu_share(CAFE)["cum_share"].iloc[-1] == 100.0)
check("top80", lambda: top80(CAFE) == ["아메리카노", "카페라떼", "바닐라라떼", "콜드브루", "레몬에이드", "녹차라떼", "베이글"])
tiny = pd.DataFrame({"date": ["2025-03-03"] * 3, "time": ["09:00"] * 3, "menu": ["A", "B", "C"],
                     "unit_price": [800, 100, 100], "qty": [1, 1, 1]})
check("정확히 80%면 거기서 멈춤", lambda: top80(tiny) == ["A"])

---
## 문제 4. 기온 구간 분석 (AI 협업, 3점)

`temp_bands(cafe, weather)`는 영업일마다 매출·아이스 비율을 구하고 기온과 합쳐, **기온 구간별 요약표**를 돌려줍니다.

1. 일별 `revenue`(매출 합계), `ice_ratio` = ICE 판매 수량 ÷ (ICE + HOT 판매 수량) (`temp_type`이 `"-"`인 디저트는 제외)
2. 날씨의 `avg_temp`를 날짜로 합칩니다. **합친 뒤 영업일 수가 그대로인지** 확인하고, 다르면 `ValueError`
3. 구간: `10℃ 미만`, `10~15`, `15~20`, `20~25`, `25℃ 이상` (`pd.cut`, 경계: −∞, 10, 15, 20, 25, ∞ / 오른쪽 경계 포함 기본값)
4. 결과 열: `days`(일수), `revenue`(일평균 매출, 정수 반올림), `ice_pct`(평균 아이스 비율 %, 소수 첫째 자리)

**AI 협업 과제**: AI에게 이 분석을 요청하면서 **"상관을 인과로 해석하지 말라"**, **"일평균으로 비교하라"** 를 프롬프트에 넣었는지 기록하세요. 그리고 AI가 쓴 해석 문장 하나를 결과표로 검증해 아래 기록에 적습니다.

In [ ]:
def temp_bands(cafe, weather):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
check("구간 5개 순서", lambda: list(temp_bands(CAFE, WEATHER).index) == ["10℃ 미만", "10~15", "15~20", "20~25", "25℃ 이상"])
check("일수 합계 = 영업일 95", lambda: temp_bands(CAFE, WEATHER)["days"].sum() == 95)
check("아이스 비율이 구간마다 증가", lambda: temp_bands(CAFE, WEATHER)["ice_pct"].is_monotonic_increasing)
check("25℃ 이상 행", lambda: temp_bands(CAFE, WEATHER).loc["25℃ 이상"].tolist() == [6, 168017, 81.3])
dup_weather = pd.concat([WEATHER, WEATHER.iloc[[70]]])                 # 날짜가 중복된 날씨 표


def _raises(exc, fn, *args):
    try:
        fn(*args)
    except exc:
        return True
    return False

check("병합으로 행이 늘면 ValueError", lambda: _raises(ValueError, temp_bands, CAFE, dup_weather))

### AI 활용 기록 (문제 4)

| 항목 | 내용 |
|---|---|
| 사용한 AI 도구 |  |
| 프롬프트에 넣은 해석 조건 (상관 vs 인과, 일평균 등) |  |
| AI가 쓴 해석 문장 하나 |  |
| 그 문장을 결과표로 검증한 결과 (맞음/과장/틀림과 이유) |  |

---
## 최종 프로젝트 제안서 (문제 4에 포함)

다음 형식으로 작성합니다. 13주차에 피드백을 받고 수정합니다. 자세한 기준은 [최종 프로젝트 안내](../../projects/final-project.md)를 참고하세요.

### 제목:

**1. 연구 질문 (2~3개, 구체적이고 비교가 있는 질문)**
1.
2.

**2. 데이터** (출처·이용 조건 / 한 행의 의미 / 기간 / 주요 열)

**3. 분석 방법** (정제 계획 / 비교·집계 방법 / 예상 그래프 2~3개)

**4. 예상 결과와 한계**

**5. AI 활용 계획** (어디에 쓰고 어떻게 검증할 것인가)

---
## 성찰 (필수)

1. "한 행의 의미"를 먼저 확인한 것이 분석에 어떤 도움이 되었나요?
2. 합계 대신 일평균으로 바꾸었을 때 결론이 달라진 예가 있었나요?
3. AI의 해석 문장을 검증하면서 배운 점은?

(여기에 작성)